# 10.4 Benchmark 启动与配置

## 本节目标

- 使用原 README 的启动入口
- 理解矩阵输入回退顺序
- 管理 warmup、repeat 和结果路径

## 环境检查

直接检查 Ascend NPU 与 CANN 环境，并记录 DEVICE_ID、矩阵、warmup、repeat 和容差。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("说明：本章默认运行 Xyce Adapter benchmark，不宣称完整 upstream Xyce 仿真。")


## 启动入口

`scripts/run.sh` 在需要时触发构建，然后执行 `build/bin/xyce_benchmark`。默认 warmup=0、repeat=10，矩阵目录和 CSV 都锚定工程根目录；额外 CLI 参数继续传给 benchmark。

`--warmup N` 只丢弃 N 次额外 solver 调用：矩阵在进入 `run_solver_repeated` 前已读取/生成一次，wrapper 在重复循环外创建；每次 `app.run` 新建 `XyceLinearSolverAdapter`，Device prepare 初始化 communicator，`distributed_gmres` 重建 solver 内的 RTC/Device 状态——不是每次重新从磁盘完整加载矩阵，也不属于完整进程冷启动；`--warmup` 不是 warm cache 预热。课程推荐 `--warmup 0`（或 `ASCEND_XYCE_WARMUP=0`）。

矩阵输入按三层回退：当前 `matrices/*.csrbin` → 相邻 Ascend-GMRES matrices → 固定种子 synthetic CSR。课程副本不依赖相邻目录，因此首次运行会生成兼容矩阵缓存。

## 快速运行一个输入

下面只运行 U1 并缩短迭代次数，用于验证部署链路。正式历史结果使用 `bash scripts/run.sh` 的 3/10 配置。

In [ ]:
%%bash
set -e
cd src/ascend_xyce
ASCEND_XYCE_WARMUP=0 ASCEND_XYCE_REPEAT=1 bash scripts/run.sh --matrix U1 --csv results/course_u1.csv
head -4 results/course_u1.csv

## 成功判据

进程退出码为 0；三个 solver 均 `converged=1`；final residual 满足阈值；CSV 含三行 solver 数据。这里没有 rank/device 参数，因为该工程不是 MPI/HCCL 多进程作业。

## 课后实践

为 U2 写出正式启动命令，并列出输入、输出和成功判据。参考答案见 `answer/10.04_answer.md`。

## 实验记录与练习

CPU 与 NPU 对比必须保持输入和参数一致，并确认输出为 Ascend C RTC Device GMRES。

完成后回答：实际后端是什么？reference 与 tolerance 是什么？主要耗时来自计算、通信、传输还是同步？改变一个并发或算法参数后，正确性和性能如何变化？参考答案仅通过本章 `answer/` 链接查阅。
